# Stage 6. Evaluation and Ablation

Notebook ini menghitung metrik klinis lengkap pada prediksi out-of-fold model produksi terpilih (Bland-Altman, ROC/AUC dengan threshold Youden, PPV/NPV, Cohen's kappa severity), lalu menjalankan ablation komponen arsitektur satu per satu. Model produksi terpilih adalah path_b_deep (fitur deep ResNet18-CSA saja, tanpa fitur hand-crafted), dipilih pada Stage 5 berdasarkan F1 tertinggi (0.621) dan recall tertinggi (0.889) di antara tujuh konfigurasi, bukan berdasarkan AUC semata karena AUC tidak mencerminkan performa pada ambang keputusan default untuk kelas anemic yang timpang (32.6 persen). Karena path_b_deep tidak memakai fitur hand-crafted maupun fusion attention, ablation Part B berfokus pada komponen yang relevan untuk model ini yaitu CSA, demografi, site token, dan dual loss. Uji cross-dataset generalization tidak dilakukan karena nail hanya berasal dari satu populasi (Valles-Coral), berbeda dari situs konjungtiva yang punya dua populasi.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, cohen_kappa_score, confusion_matrix, f1_score

from configs import paths
from src.common import eval as evaluation
from src.common import features, manifest as manifest_utils, train

output_dir = paths.outputs_dir("nail")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)

handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")
deep_embeddings = np.load(output_dir / "deep_embeddings.npy")
embedding_uids = pd.read_csv(output_dir / "deep_embeddings_uids.csv")["uid"].tolist()
oof = pd.read_csv(output_dir / "multitask_oof_best_f1.csv")

print("oof rows", len(oof))

# Part A. Deep Clinical Evaluation

Bagian ini memakai prediksi out-of-fold model path_b_deep dari Stage 5, tanpa pelatihan ulang.

## Bland-Altman Analysis for Hemoglobin Regression

Bias mendekati nol dan limits of agreement yang sempit menandakan prediksi Hb tidak bias secara sistematis terhadap nilai laboratorium.

In [ ]:
regression_metrics = evaluation.regression_summary(oof["hb_true"], oof["hb_pred"])
bland_altman = evaluation.bland_altman_stats(oof["hb_true"], oof["hb_pred"])
print("MAE", round(regression_metrics["mae"], 4))
print("RMSE", round(regression_metrics["rmse"], 4))
print("R squared", round(regression_metrics["r_squared"], 4))
print("bias", round(bland_altman["bias"], 4))
print("limits of agreement", round(bland_altman["lower_limit"], 4), "sampai", round(bland_altman["upper_limit"], 4))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(bland_altman["mean_value"], bland_altman["difference"], s=10, alpha=0.5)
ax.axhline(bland_altman["bias"], color="black", label="bias")
ax.axhline(bland_altman["lower_limit"], color="red", linestyle="--", label="limits of agreement")
ax.axhline(bland_altman["upper_limit"], color="red", linestyle="--")
ax.set_xlabel("Mean of true and predicted Hb (g/dL)")
ax.set_ylabel("Predicted minus true Hb (g/dL)")
ax.set_title("Bland-Altman Plot")
ax.legend()
plt.tight_layout()
plt.show()

## ROC Curve and Sensitivity-Prioritized Threshold

Skrining anemia mengutamakan sensitivitas agar kasus positif tidak terlewat. Threshold Youden dicari sebagai alternatif dari threshold default 0.5 yang dipakai pada Stage 5.

In [ ]:
y_true = oof["anemic_true"].to_numpy()
y_prob = oof["anemic_prob"].to_numpy().astype(np.float64)

false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_prob)
roc_auc = auc(false_positive_rate, true_positive_rate)
youden = evaluation.youden_optimal_threshold(y_true, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(false_positive_rate, true_positive_rate, label=f"AUC = {roc_auc:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve, Anemia Classification")
ax.legend()
plt.tight_layout()
plt.show()

print("AUC", round(roc_auc, 4))
print("Youden threshold", round(youden["threshold"], 4))
print("Youden sensitivity", round(youden["sensitivity"], 4), "specificity", round(youden["specificity"], 4))

In [ ]:
for label, threshold in [("default 0.5", 0.5), ("Youden", youden["threshold"])]:
    prediction = (y_prob >= threshold).astype(int)
    ppv_npv = evaluation.ppv_npv(y_true, prediction)
    f1 = f1_score(y_true, prediction)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")
    print(label)
    print("  sensitivity", round(sensitivity, 4), "specificity", round(specificity, 4))
    print("  PPV", round(ppv_npv["ppv"], 4), "NPV", round(ppv_npv["npv"], 4), "F1", round(f1, 4))

## Severity Confusion Matrix and Cohen's Kappa

Evaluasi tiga kelas severity (Non-Anemic, Mild, Moderate) yang tersedia pada dataset nail, hanya pada baris dengan label severity valid. Kelas Moderate minoritas keras (8.8 persen), sesuai disclaimer eksperimen sekunder pada Stage 5.

In [ ]:
severity_valid = oof["severity_true"] >= 0
severity_true = oof.loc[severity_valid, "severity_true"]
severity_pred = oof.loc[severity_valid, "severity_pred"]

kappa = cohen_kappa_score(severity_true, severity_pred)
print("Cohen kappa (severity)", round(kappa, 4))
print("confusion matrix (baris label sebenarnya, kolom prediksi):")
print(confusion_matrix(severity_true, severity_pred))

# Part B. Component Ablation

Setiap komponen dimatikan satu per satu dari konfigurasi path_b_deep, memakai protokol pelatihan identik dengan Stage 5 (epochs 60, tanpa fitur hand-crafted maupun fusion attention karena model produksi memang hanya memakai jalur deep). CSA diablasi lewat ekstraksi ulang embedding tanpa modul channel spatial attention, sedangkan demografi, site token, dan dual loss diablasi lewat parameter run_kfold.

In [ ]:
backbone_no_csa = features.EmbeddingBackbone(backbone_name="resnet18", use_csa=False)
embeddings_no_csa, embedding_uids_no_csa = features.extract_deep_embeddings(manifest, model=backbone_no_csa)
print("embeddings without CSA", embeddings_no_csa.shape)

In [ ]:
def run_path_b_deep(deep_embeddings_override=None, embedding_uids_override=None, **overrides):
    embeddings_used = deep_embeddings if deep_embeddings_override is None else deep_embeddings_override
    embedding_uids_used = embedding_uids if embedding_uids_override is None else embedding_uids_override
    kwargs = dict(use_handcrafted=False, use_deep=True)
    kwargs.update(overrides)
    return train.run_kfold(manifest, handcrafted, embeddings_used, embedding_uids_used, n_splits=5, epochs=60, **kwargs)


ablation_results = {}
ablation_results["CSA off"] = run_path_b_deep(deep_embeddings_override=embeddings_no_csa, embedding_uids_override=embedding_uids_no_csa)
ablation_results["Dual Loss off (MSE only)"] = run_path_b_deep(regression_loss="mse_only")
ablation_results["Site token off"] = run_path_b_deep(use_site_token=False)
ablation_results["Demographics off"] = run_path_b_deep(use_demographics=False)

print("selesai empat run ablation")

In [ ]:
baseline_fold_metrics = pd.read_csv(output_dir / "multitask_model_comparison.csv")
baseline_row = baseline_fold_metrics[baseline_fold_metrics["configuration"] == "path_b_deep"].iloc[0]

ablation_rows = [{
    "configuration": "path_b_deep (baseline)",
    "mae_mean": baseline_row["mae"],
    "accuracy_mean": baseline_row["accuracy"],
    "f1_mean": baseline_row["f1"],
    "auc_mean": baseline_row["auc"],
}]
for name, result in ablation_results.items():
    metrics = result["fold_metrics"]
    ablation_rows.append({
        "configuration": name,
        "mae_mean": metrics["mae"].mean(),
        "accuracy_mean": metrics["accuracy"].mean(),
        "f1_mean": metrics["f1"].mean(),
        "auc_mean": metrics["auc"].mean(),
    })
ablation_table = pd.DataFrame(ablation_rows)
print(ablation_table.round(4).to_string(index=False))
ablation_table.to_csv(output_dir / "ablation_component_comparison.csv", index=False)